# D2.3 · Scoping an agentic incident

**Function D — Security Operations → The Incident Responder**  ·  *Security of AI*

Builds on **[D2.2 · When the actor is an agent](https://spbreed.github.io/cyber-commons/lessons/D2.2.html)**.

| | |
|---|---|
| Open-source tooling | OpenTelemetry |
| Open-weight models | Kimi K2 |

> **Runs anywhere.** Every line of code is in this notebook — nothing to install, nothing to clone, no API key, no network. Standard library only, so it works on a Kaggle kernel with the internet switched off.

## 1 · The concept


Scoping answers "what was touched?" For a host-based incident you enumerate
hosts. For an agentic incident, **scope follows the delegation graph**.

The agent that touched the resource is usually the *last* actor in a chain. If
you scope only that actor, you miss everything the earlier actors reached — and
because authority narrows down the chain, the earlier actors typically had
*more* access, not less.

The undercount is systematic and it grows with delegation depth, which is the
operational reason B2.6 bounds depth in the first place.

## 2 · Demo — scope the chain, not the actor

In [ ]:
REACHED = {
 "dana@corp":    ["repo-core", "repo-infra", "vault-dev"],
 "orchestrator": ["repo-core", "queue-tasks"],
 "patch-agent":  ["repo-core", "repo-payments"],
 "deploy-agent": ["cluster-prod"],
}
CHAIN = ["dana@corp", "orchestrator", "patch-agent", "deploy-agent"]

def scope(chain, reached):
    last_only = set(reached.get(chain[-1], []))
    full = {r for a in chain for r in reached.get(a, [])}
    return {"chain": " → ".join(chain),
            "scoped_last_actor_only": sorted(last_only),
            "scoped_whole_chain": sorted(full),
            "missed_by_naive_scoping": sorted(full - last_only),
            "undercount_factor": round(len(full)/len(last_only), 2) if last_only else None}

s = scope(CHAIN, REACHED)
for k, v in s.items(): print(f"{k:26s}{v}")
print("\nScoping the last actor finds one cluster. The chain reached six")
print("resources, including a payments repository and a dev vault.")

## 3 · Where it breaks — the undercount grows with depth

In [ ]:
print(f"{'depth':>6}{'last-actor scope':>19}{'chain scope':>14}{'undercount':>12}")
print("-" * 52)
for d in range(1, 5):
    sub = CHAIN[:d]
    r = scope(sub, REACHED)
    print(f"{d:>6}{len(r['scoped_last_actor_only']):>19}"
          f"{len(r['scoped_whole_chain']):>14}"
          f"{str(r['undercount_factor']):>12}")
print("\nEach hop adds resources the last actor never touched. This is why B2.6")
print("bounds delegation depth: depth is an incident-scope multiplier.")

## 4 · The control — scope from the act chain, then widen by shared resources

In [ ]:
SHARED = {"repo-core": ["build-agent", "test-agent"],
          "cluster-prod": ["deploy-agent", "monitor-agent"],
          "repo-payments": ["finance-agent"]}

def scope_transitive(chain, reached, shared, hops=1):
    """Anything that shares a touched resource may have been influenced."""
    direct = {r for a in chain for r in reached.get(a, [])}
    exposed = set(chain)
    frontier = set(direct)
    for _ in range(hops):
        nxt = set()
        for res in frontier:
            for actor in shared.get(res, []):
                if actor not in exposed:
                    exposed.add(actor)
                    nxt |= set(reached.get(actor, []))
        frontier = nxt
    return {"resources_direct": sorted(direct),
            "actors_in_scope": sorted(exposed),
            "second_order_actors": sorted(exposed - set(chain))}

t = scope_transitive(CHAIN, REACHED, SHARED)
for k, v in t.items(): print(f"{k:22s}{v}")
print("\nFive more identities shared a resource with the compromised chain.")
print("They are not confirmed compromised — they are IN SCOPE, which is different")
print("and is the distinction an incident record has to make explicitly.")
assert t["second_order_actors"]

In [ ]:
# Verify: produce the scope statement for the incident record.
def scope_statement(chain, reached, shared):
    s = scope(chain, reached)
    t = scope_transitive(chain, reached, shared)
    return (f"SCOPE\n"
            f"  chain              {s['chain']}\n"
            f"  confirmed touched  {s['scoped_whole_chain']}\n"
            f"  would have been missed by scoping the acting agent alone:\n"
            f"                     {s['missed_by_naive_scoping']}\n"
            f"  undercount factor  {s['undercount_factor']}×\n"
            f"  in scope, not confirmed (shared a resource):\n"
            f"                     {t['second_order_actors']}")
print(scope_statement(CHAIN, REACHED, SHARED))

## What you just proved

Scoping the last actor finds `cluster-prod` alone; the whole chain reaches six resources, missing five, with an undercount factor of 6.0. The undercount grows with each hop. Transitive scoping adds five second-order identities that shared a resource, explicitly marked as in scope rather than confirmed compromised.

## Your turn

For your last incident involving a service account, recompute the scope by walking what else that account could reach. The number is almost always larger than what was written in the report.

---

**Next → [D2.4 · Containment at machine speed](https://spbreed.github.io/cyber-commons/lessons/D2.4.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/D2.3.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/D2.3.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*